%md
#Notebook para  Capa Gold

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
%sql
create WIDGET text storageName default "adventureworksltadlsd";
CREATE WIDGET TEXT catalogName DEFAULT 'catadvwd';
CREATE WIDGET TEXT silverSchemaName DEFAULT 'silver';
CREATE WIDGET TEXT goldSchemaName DEFAULT 'golden';


In [0]:
catalogo = dbutils.widgets.get("catalogName")
silverSchema = dbutils.widgets.get("silverSchemaName")
goldSchema = dbutils.widgets.get("goldSchemaName")

In [0]:
#sobrescribimos, porque Gold se recalcula desde cero en cada corrida.

def overwrite_to_gold(df, nombre_tabla):
    tabla_destino = f"{catalogo}.{goldSchema}.{nombre_tabla}"
    try:
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(tabla_destino)
        print(f"{tabla_destino} recalculada ({df.count()} filas)")
    except Exception as e:
        print(f"Error al cargar {tabla_destino}: {e}")
        raise

In [0]:
df_customer_silver = spark.table(f"{catalogo}.{silverSchema}.customers")
 
dim_customer = df_customer_silver.select(
    col("customerId"),
    col("title"),
    col("firstName"),
    col("middleName"),
    col("lastName"),
    col("suffix"),
    col("companyName"),
    col("salesPerson"),
    col("emailAddress"),
    col("phone"))


In [0]:
overwrite_to_gold(dim_customer, "dimcustomer")

catadvwd.golden.dimcustomer recalculada (847 filas)


In [0]:
df_product_silver = spark.table(f"{catalogo}.{silverSchema}.products")
 
dim_product = df_product_silver.select(
    col("productId"),
    col("productName"),
    col("productNumber"),
    col("color"),
    col("standardCost"),
    col("listPrice"),
    col("size"),
    col("weight"),
    col("sellStartDate"),
    col("sellEndDate"),
    col("discontinuedDate"),
    col("productCategoryName"),
    col("productModelName")
)

In [0]:
overwrite_to_gold(dim_product, "dimproduct")

catadvwd.golden.dimproduct recalculada (295 filas)


Unimos `salesorderdetail` (que define el grano) con `salesorderheader` (para traer `orderDate`, `status`, `salesOrderNumber` y las medidas de encabezado, que —recordemos— se repetirán por línea).

Usamos `inner join` a propósito. Si una línea de detalle no tiene encabezado correspondiente (o viceversa), es una inconsistencia de integridad referencial que **no** queremos que llegue silenciosamente a Gold — mejor que esa línea desaparezca del fact (y la detectes) que reportarla con un encabezado inexistente.

In [0]:
df_detail_silver = spark.table(f"{catalogo}.{silverSchema}.salesorderdetail")
df_header_silver = spark.table(f"{catalogo}.{silverSchema}.salesorderheader")
 
fact_sales = (
    df_detail_silver.alias("det")
    .join(
        df_header_silver.alias("hdr"),
        col("det.salesOrderId") == col("hdr.salesOrderId"),
        "inner"
    )
    .select(
        # Llaves
        col("det.salesOrderDetailId"),
        col("det.salesOrderId"),
        col("hdr.customerId"),
        col("det.productId"),
        # Atributos de encabezado (útiles para filtrar/reportear)
        col("hdr.orderDate"),
        col("hdr.status"),
        col("hdr.salesOrderNumber"),
        # Métricas de línea (aditivas al grano del fact)
        col("det.orderQty"),
        col("det.unitPrice"),
        col("det.unitPriceDiscount"),
        col("det.lineTotal"),
        # Métricas de encabezado (semi-aditivas: se repiten por línea, ver nota arriba)
        col("hdr.subTotal"),
        col("hdr.taxAmt"),
        col("hdr.freight"),
        col("hdr.totalDue")
    )
)
 

In [0]:
overwrite_to_gold(fact_sales, "factsales")

catadvwd.golden.factsales recalculada (542 filas)


In [0]:
for tabla in ["dimcustomer", "dimproduct", "factsales"]:
    conteo = spark.table(f"{catalogo}.{goldSchema}.{tabla}").count()
    print(f"{catalogo}.{goldSchema}.{tabla}: {conteo} filas")

catadvwd.golden.dimcustomer: 847 filas
catadvwd.golden.dimproduct: 295 filas
catadvwd.golden.factsales: 542 filas


In [0]:
%sql
select  * from catadvwd.golden.dimcustomer limit(10)

customerId,title,firstName,middleName,lastName,suffix,companyName,salesPerson,emailAddress,phone
1,Mr.,Orlando,N.,Gee,null,A Bike Store,adventure-works\pamela0,orlando0@adventure-works.com,245-555-0173
2,Mr.,Keith,null,Harris,null,Progressive Sports,adventure-works\david8,keith0@adventure-works.com,170-555-0127
3,Ms.,Donna,F.,Carreras,null,Advanced Bike Components,adventure-works\jillian0,donna0@adventure-works.com,279-555-0130
4,Ms.,Janet,M.,Gates,null,Modular Cycle Systems,adventure-works\jillian0,janet1@adventure-works.com,710-555-0173
5,Mr.,Lucy,null,Harrington,null,Metropolitan Sports Supply,adventure-works\shu0,lucy0@adventure-works.com,828-555-0186
6,Ms.,Rosmarie,J.,Carroll,null,Aerobic Exercise Company,adventure-works\linda3,rosmarie0@adventure-works.com,244-555-0112
7,Mr.,Dominic,P.,Gash,null,Associated Bikes,adventure-works\shu0,dominic0@adventure-works.com,192-555-0173
10,Ms.,Kathleen,M.,Garza,null,Rural Cycle Emporium,adventure-works\josé1,kathleen0@adventure-works.com,150-555-0127
11,Ms.,Katherine,null,Harding,null,Sharp Bikes,adventure-works\josé1,katherine0@adventure-works.com,926-555-0159
12,Mr.,Johnny,A.,Caprio,Jr.,Bikes and Motorbikes,adventure-works\garrett1,johnny0@adventure-works.com,112-555-0191


In [0]:
%sql
select * from catadvwd.golden.dimproduct  limit (10)

productId,productName,productNumber,color,standardCost,listPrice,size,weight,sellStartDate,sellEndDate,discontinuedDate,productCategoryName,productModelName
680,"HL Road Frame - Black, 58",FR-R92B-58,Black,10593100.00,14315000.00,58,1016.040,2002-06-01T00:00:00Z,null,null,Road Frames,HL Road Frame
706,"HL Road Frame - Red, 58",FR-R92R-58,Red,10593100.00,14315000.00,58,1016.040,2002-06-01T00:00:00Z,null,null,Road Frames,HL Road Frame
707,"Sport-100 Helmet, Red",HL-U509-R,Red,130863.00,349900.00,null,null,2005-07-01T00:00:00Z,null,null,Helmets,Sport-100
708,"Sport-100 Helmet, Black",HL-U509,Black,130863.00,349900.00,null,null,2005-07-01T00:00:00Z,null,null,Helmets,Sport-100
709,"Mountain Bike Socks, M",SO-B909-M,White,33963.00,95000.00,M,null,2005-07-01T00:00:00Z,2006-06-30T00:00:00Z,null,Socks,Mountain Bike Socks
710,"Mountain Bike Socks, L",SO-B909-L,White,33963.00,95000.00,L,null,2005-07-01T00:00:00Z,2006-06-30T00:00:00Z,null,Socks,Mountain Bike Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,Blue,130863.00,349900.00,null,null,2005-07-01T00:00:00Z,null,null,Helmets,Sport-100
712,AWC Logo Cap,CA-1098,Multi,69223.00,89900.00,null,null,2005-07-01T00:00:00Z,null,null,Caps,Cycling Cap
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,Multi,384923.00,499900.00,S,null,2005-07-01T00:00:00Z,null,null,Jerseys,Long-Sleeve Logo Jersey
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,Multi,384923.00,499900.00,M,null,2005-07-01T00:00:00Z,null,null,Jerseys,Long-Sleeve Logo Jersey


In [0]:
%sql
select * from catadvwd.golden.factsales  limit (10)

salesOrderDetailId,salesOrderId,customerId,productId,orderDate,status,salesOrderNumber,orderQty,unitPrice,unitPriceDiscount,lineTotal,subTotal,taxAmt,freight,totalDue
110562,71774,29847,836,2008-06-01T00:00:00Z,5,SO71774,1,3568980.00,0.00,356.90,8803484.000,704279.000,220087.000,9727850.000
110563,71774,29847,822,2008-06-01T00:00:00Z,5,SO71774,1,3568980.00,0.00,356.90,8803484.000,704279.000,220087.000,9727850.000
110567,71776,30072,907,2008-06-01T00:00:00Z,5,SO71776,1,639000.00,0.00,63.90,788100.000,63048.000,19703.000,870851.000
110616,71780,30113,905,2008-06-01T00:00:00Z,5,SO71780,4,2184540.00,0.00,873.82,null,null,9604672.000,null
110617,71780,30113,983,2008-06-01T00:00:00Z,5,SO71780,2,4616940.00,0.00,923.39,null,null,9604672.000,null
110618,71780,30113,988,2008-06-01T00:00:00Z,5,SO71780,6,1129980.00,4000.00,406.79,null,null,9604672.000,null
110619,71780,30113,748,2008-06-01T00:00:00Z,5,SO71780,2,8187000.00,0.00,1637.40,null,null,9604672.000,null
110620,71780,30113,990,2008-06-01T00:00:00Z,5,SO71780,1,3239940.00,0.00,323.99,null,null,9604672.000,null
110621,71780,30113,926,2008-06-01T00:00:00Z,5,SO71780,1,1498740.00,0.00,149.87,null,null,9604672.000,null
110622,71780,30113,743,2008-06-01T00:00:00Z,5,SO71780,1,8097600.00,0.00,809.76,null,null,9604672.000,null


In [0]:
df= "catadvwd.golden.dimcustomer"

In [0]:
display(spark.table(df).describe())

summary,customerId,title,firstName,middleName,lastName,suffix,companyName,salesPerson,emailAddress,phone
count,847,840,847,504,847,48,847,847,847,847
mean,14503.217237308147,null,null,null,null,null,null,null,null,null
stddev,14729.700946283214,null,null,null,null,null,null,null,null,null
min,1,Mr.,A.,A.,Abel,II,A Bike Store,adventure-works\david8,a0@adventure-works.com,1 (11) 500 555-0110
max,30118,Sra.,Yvonne,Z.,Vicknair,Sr.,Yellow Bicycle Company,adventure-works\shu0,yvonne1@adventure-works.com,996-555-0196


In [0]:
tables = [
    "catadvwd.golden.dimcustomer",
    "catadvwd.golden.dimproduct",
    "catadvwd.golden.factsales"
]

md_list = ""
for table in tables:
    df = spark.table(table)
    md_list += f"\n### {table}\n"
    md_list += "\n- Columnas:\n"
    for field in df.schema.fields:
        md_list += f"  - `{field.name}`: {field.dataType.simpleString()}\n"

displayHTML(md_list)

### catadvwd.golden.dimcustomer

- Columnas:
 - `customerId`: int
 - `title`: string
 - `firstName`: string
 - `middleName`: string
 - `lastName`: string
 - `suffix`: string
 - `companyName`: string
 - `salesPerson`: string
 - `emailAddress`: string
 - `phone`: string

### catadvwd.golden.dimproduct

- Columnas:
 - `productId`: int
 - `productName`: string
 - `productNumber`: string
 - `color`: string
 - `standardCost`: decimal(10,2)
 - `listPrice`: decimal(10,2)
 - `size`: string
 - `weight`: decimal(10,3)
 - `sellStartDate`: timestamp
 - `sellEndDate`: timestamp
 - `discontinuedDate`: timestamp
 - `productCategoryName`: string
 - `productModelName`: string

### catadvwd.golden.factsales

- Columnas:
 - `salesOrderDetailId`: int
 - `salesOrderId`: int
 - `customerId`: int
 - `productId`: int
 - `orderDate`: timestamp
 - `status`: int
 - `salesOrderNumber`: string
 - `orderQty`: int
 - `unitPrice`: decimal(13,2)
 - `unitPriceDiscount`: decimal(13,2)
 - `lineTotal`: decimal(13,2)
 - `subTotal`: decimal(10,3)
 - `taxAmt`: decimal(10,3)
 - `freight`: decimal(10,3)
 - `totalDue`: decimal(10,3)

In [0]:
%sql
select b.productName, count(1)  from catadvwd.golden.factsales  as a
inner join catadvwd.golden.dimproduct as b
on a.productId=b.productId
where a.productId=836
group by b.productName
 limit (10)

productName,count(1)
"ML Road Frame-W - Yellow, 48",4


In [0]:
%sql
select* 
from catadvwd.golden.factsales  as a
inner join catadvwd.golden.dimproduct as b
on a.productId=b.productId
where a.productId=836
--group by b.productName
 limit (10)


salesOrderDetailId,salesOrderId,customerId,productId,orderDate,status,salesOrderNumber,orderQty,unitPrice,unitPriceDiscount,lineTotal,subTotal,taxAmt,freight,totalDue,productId,productName,productNumber,color,standardCost,listPrice,size,weight,sellStartDate,sellEndDate,discontinuedDate,productCategoryName,productModelName
113089,71915,29638,836,2008-06-01T00:00:00Z,5,SO71915,3,3568980.00,0.00,1070.69,null,1709785.000,534308.000,null,836,"ML Road Frame-W - Yellow, 48",FR-R72Y-48,Yellow,3609428.00,5948300.00,48,1061.400,2006-07-01T00:00:00Z,null,null,Road Frames,ML Road Frame-W
111081,71797,29796,836,2008-06-01T00:00:00Z,5,SO71797,4,3568980.00,0.00,1427.59,null,null,null,null,836,"ML Road Frame-W - Yellow, 48",FR-R72Y-48,Yellow,3609428.00,5948300.00,48,1061.400,2006-07-01T00:00:00Z,null,null,Road Frames,ML Road Frame-W
110741,71783,29957,836,2008-06-01T00:00:00Z,5,SO71783,4,3568980.00,0.00,1427.59,null,null,null,null,836,"ML Road Frame-W - Yellow, 48",FR-R72Y-48,Yellow,3609428.00,5948300.00,48,1061.400,2006-07-01T00:00:00Z,null,null,Road Frames,ML Road Frame-W
110562,71774,29847,836,2008-06-01T00:00:00Z,5,SO71774,1,3568980.00,0.00,356.90,8803484.000,704279.000,220087.000,9727850.000,836,"ML Road Frame-W - Yellow, 48",FR-R72Y-48,Yellow,3609428.00,5948300.00,48,1061.400,2006-07-01T00:00:00Z,null,null,Road Frames,ML Road Frame-W
